In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path("../src").resolve()))

from config import RAW_ROOT, GROUP3_PATIENTS, BAD_FILES
from utils import load_raw_edf, prepare_raw

group3_valid_files = []

# collect all valid EDF files from Group 3
for patient_name in GROUP3_PATIENTS:
    patient_folder = RAW_ROOT / patient_name
    edf_files = sorted(patient_folder.glob("*.edf"))
    patient_bad_files = set(BAD_FILES.get(patient_name, []))

    for file_path in edf_files:
        if file_path.name in patient_bad_files:
            print(f"Skipping bad file: {file_path.name}")
            continue
        group3_valid_files.append(file_path)

print(f"\nTotal valid Group 3 EDF files: {len(group3_valid_files)}")

if len(group3_valid_files) == 0:
    raise ValueError("No valid Group 3 EDF files found.")

# use first valid file as reference for channel order
first_raw = load_raw_edf(group3_valid_files[0])
first_raw = prepare_raw(first_raw)
reference_channels = first_raw.ch_names.copy()

print("\nReference file:")
print(group3_valid_files[0].name)
print("Reference channels:")
print(reference_channels)

# start intersection
common_channels = set(reference_channels)

for file_path in group3_valid_files[1:]:
    raw = load_raw_edf(file_path)
    raw = prepare_raw(raw)
    current_channels = set(raw.ch_names)

    common_channels = common_channels.intersection(current_channels)

# keep the order from the first file
group3_common_channels = [ch for ch in reference_channels if ch in common_channels]

print("\nGroup 3 common channels:")
print(group3_common_channels)
print(f"\nNumber of common channels: {len(group3_common_channels)}")